## Загрузка исходных датасетов

- `load_from_cdaweb` - параметр, отвечающий за источник откуда будет произведена загрузка данных.
- `loader` - экзампляр класса, обеспичвающий загрузку данных.
- `get_*_data()` - соответственные методы по загрузке определённых данных.

| Вызов | Содержимое колонок |
|--------|---------------------|
| `get_ssc_data()` | **Time** — время; **Latitude**, **Longitude** — широта/долгота (GSM); **L** — L-шелл; **GSM_X**, **GSM_Y**, **GSM_Z** — позиция аппарата в GSM (км). |
| `get_fgm_data()` | **Time**; **GSM_Bx**, **GSM_By**, **GSM_Bz** — вектор магнитного поля в GSM (нТ). |
| `get_esa_data(particle="ion")` | **Time**; **GSM_Vix**, **GSM_Viy**, **GSM_Viz** — скорость ионов в GSM (км/с). |
| `get_esa_data(particle="electron")` | **Time**; **GSM_Vex**, **GSM_Vey**, **GSM_Vez** — скорость электронов в GSM (км/с). |
| `get_efi_data()` | **Time**; **GSM_Ex**, **GSM_Ey**, **GSM_Ez** — электрическое поле в GSM (единицы как в CDF, часто мВ/м). |
| `get_sta_data()` | **Time**; **GSM_Vsx**, **GSM_Vsy**, **GSM_Vsz** — скорость спутника в GSM (км/с). |
| `get_omn_data()` | **Time**; **FP** — динамическое давление солнечного ветра; **Bz_GSM** — компонента Bz в GSM. |
| `get_shue_data()` | **Time**; **L** — L-шелл; **MLT** — магнитное локальное время (из `Longitude`); **r** — параметр Shue (SSC+OMNI). |

Общее: после загрузки отброшены строки без валидного **Time**, дубликаты по **Time** удалены. Дополнительно в OMNI интерполируются пропуски в `FP` и `Bz_GSM` (в обе стороны), чтобы интервалы доступности не дробились из-за `None`. Интервал и спутник задаются в `config.reading`; OMNI — околоземный ряд, не привязан к букве спутника по смыслу данных.

In [ ]:
from backend.src.io.loader import DataDownloading
from backend.src.config import config
from backend.src.processing.utils.h_parameter_range import show_h_parameter_range

In [ ]:
load_from_cdaweb = False
loader = DataDownloading(config, load_from_cdaweb=load_from_cdaweb)

In [ ]:
ssc_data = loader.get_ssc_data()
fgm_data = loader.get_fgm_data()
esa_ion_data = loader.get_esa_data(particle="ion")
# esa_electron_data = loader.get_esa_data(particle="electron")
efi_data = loader.get_efi_data()
sta_data = loader.get_sta_data()
omn_data = loader.get_omn_data()
shue_data = loader.get_shue_data()

## Пересечение промежутков

In [ ]:
from datetime import timedelta

from backend.src.processing import AvailabilityIntervals
from backend.src.processing.intersections import intersect_many, summarize_intervals

In [ ]:
availability = AvailabilityIntervals(show_progress=True)

ssc_intervals = availability.from_dataframe(ssc_data, "ssc")
fgm_intervals = availability.from_dataframe(fgm_data, "fgm")
esa_ion_intervals = availability.from_dataframe(esa_ion_data, "esa_ion")
# esa_electron_intervals = availability.from_dataframe(esa_electron_data, "esa_electron")
efi_intervals = availability.from_dataframe(efi_data, "efi")
sta_intervals = availability.from_dataframe(sta_data, "sta")
shue_intervals = availability.from_dataframe(shue_data, "shue")

In [ ]:
interval_intersections = intersect_many(
    interval_groups=[
        ssc_intervals,
        sta_intervals,
        efi_intervals,
        fgm_intervals,
        esa_ion_intervals,
        shue_intervals
    ],
    min_duration=timedelta(hours=1),
)

summarize_intervals(interval_intersections)

## Визуализация пересечений

In [ ]:
# availability.show(esa_ion_data, esa_ion_intervals, "esa_ion")

In [ ]:
intervals_list = [
    {'intervals': ssc_intervals, "data_type": "ssc"},
    {'intervals': fgm_intervals, "data_type": "fgm"},
    {'intervals': esa_ion_intervals, "data_type": "esa_ion"},
    {'intervals': efi_intervals, "data_type": "efi"},
    {'intervals': sta_intervals, "data_type": "sta"},
    {'intervals': interval_intersections, "data_type": "intersections"},
]

availability.show_intervals(ssc_data, intervals_list)

## Модель Shue 1997 года

In [ ]:
from importlib import reload

from backend.src.processing import availability_intervals
from backend.src.processing.utils import intervals_view

reload(intervals_view)
reload(availability_intervals)

from backend.src.processing.shue_bordered_intervals import (
    BuildShueBorderedIntervalsIn,
    build_shue_bordered_intervals,
)

result = build_shue_bordered_intervals(
    data=BuildShueBorderedIntervalsIn(
        ssc_data=ssc_data,
        omn_data=omn_data,
        min_l=4.0,
    )
)

bordered_dataset = result.bordered_dataset
shue_intervals = result.intervals

shue_intervals

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plot_df = shue_data.copy()

if plot_df.empty:
    raise ValueError("shue_data пуст — нечего рисовать (проверьте OMNI/SSC, мэчинг по времени и фильтр L>=4 & L<=r)")

plot_df["Time"] = pd.to_datetime(plot_df["Time"], utc=True, errors="coerce").dt.tz_localize(None)
plot_df = plot_df.dropna(subset=["Time", "L", "r"]).sort_values("Time")

fig, ax = plt.subplots(1, 1, figsize=(18, 6), layout="constrained", sharex=True)
ax.plot(plot_df["Time"], plot_df["L"], label="L", linewidth=1.2)
ax.plot(plot_df["Time"], plot_df["r"], label="r (Shue)", linewidth=1.2)

ax.set_xlabel("Time")
ax.set_ylabel("RE")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()